# 04 — Content Pipeline
Explore → Extract → Analyse → Ingest → Verify

This notebook covers the **content** object type from the raw Backyard dump.

**Graph model:**
- `:Content` node
- `:SitePageCategory` node (from `content_page_category_id` / `content_page_category_name`)
- **Relationships:**
  - `(Content)-[:BELONGS_TO_SITE]->(Site)`
  - `(People)-[:CREATED]->(Content)`
  - `(People)-[:LAST_MODIFIED]->(Content)`
  - `(SitePageCategory)-[:HAS_CONTENT]->(Content)`
  - `(SitePageCategory)-[:BELONGS_TO_SITE]->(Site)`
  - `(Content)-[:HAS_FILE]->(File)` from union of file linkage signals

Assumes pipelines 01 (People), 02 (Site), and 03 (File) are already loaded.

## 0 · Setup

In [1]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_FILE, NORMALIZED_DIR, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD

print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw file          : {RAW_FILE}")
print(f"Raw file exists   : {RAW_FILE.exists()}")
print(f"Normalized dir    : {NORMALIZED_DIR}")
print(f"Normalized exists : {NORMALIZED_DIR.exists()}")

Project root      : /home/ubuntu/graph_experiments
Raw file          : /home/ubuntu/graph_experiments/data/raw/backyard_dump_19022026_172648.jsonl
Raw file exists   : True
Normalized dir    : /home/ubuntu/graph_experiments/data/normalized
Normalized exists : True


## 1 · Explore Raw Data

In [2]:
# Load all relevant object types for cross-reference checks
content_items = []
people_ids = set()
site_ids = set()
raw_all_file_ids = set()
raw_image_file_ids = set()
raw_non_image_file_ids = set()

with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        ot = obj.get("object_type")

        if ot == "content":
            content_items.append(obj)
        elif ot == "people" and obj.get("id"):
            people_ids.add(obj["id"])
        elif ot == "site" and obj.get("id"):
            site_ids.add(obj["id"])
        elif ot == "file" and obj.get("id"):
            file_id = obj["id"]
            raw_all_file_ids.add(file_id)
            if obj.get("file_is_image") is True:
                raw_image_file_ids.add(file_id)
            else:
                raw_non_image_file_ids.add(file_id)

print(f"Content records           : {len(content_items)}")
print(f"People IDs (raw)          : {len(people_ids)}")
print(f"Site IDs (raw)            : {len(site_ids)}")
print(f"All File IDs (raw)        : {len(raw_all_file_ids)}")
print(f"Image File IDs (raw)      : {len(raw_image_file_ids)}")
print(f"Non-image File IDs (raw)  : {len(raw_non_image_file_ids)}")

Content records           : 3142
People IDs (raw)          : 954
Site IDs (raw)            : 45
All File IDs (raw)        : 7877
Image File IDs (raw)      : 7322
Non-image File IDs (raw)  : 555


In [3]:
# Save 10 sample content records for debugging
sample_path = PROJECT_ROOT / "data" / "debug" / "sample_contents.jsonl"
sample_path.parent.mkdir(parents=True, exist_ok=True)
with open(sample_path, "w") as out:
    for item in content_items[:10]:
        out.write(json.dumps(item) + "\n")
print(f"Saved {min(10, len(content_items))} samples → {sample_path}")

# Pretty-print one sample (truncate heavy fields)
sample = dict(content_items[0])
for heavy_key in ["content_topics", "_aggregated_locations", "content_files"]:
    if heavy_key in sample and isinstance(sample[heavy_key], list) and len(sample[heavy_key]) > 2:
        sample[heavy_key] = sample[heavy_key][:2] + ["...[truncated]..."]
print("\n--- Sample content record ---")
print(json.dumps(sample, indent=2))

Saved 10 samples → /home/ubuntu/graph_experiments/data/debug/sample_contents.jsonl

--- Sample content record ---
{
  "id": "03e5a566-4785-4e48-8b07-23698ff5cf11",
  "autocomplete": "Add-On Products: Recognition Best Practice",
  "content_title": "Add-On Products: Recognition Best Practice",
  "content_type": "page",
  "content_site_id": "906512a7-0e25-40f5-9ace-375b5a7b59e5",
  "content_site_name": "Customer Success Managers",
  "content_site_img_url": "/content/o/b8242631-a6c6-4867-a714-ebdd8b46abc3",
  "content_site_type": "public",
  "content_site_has_pages": true,
  "content_site_has_albums": true,
  "content_site_has_events": true,
  "content_site_is_active": true,
  "content_site_is_deleted": false,
  "content_body": "{\"type\":\"doc\",\"content\":[{\"type\":\"heading\",\"attrs\":{\"indentation\":0,\"textAlign\":\"left\",\"level\":3,\"className\":\"\",\"id\":null},\"content\":[{\"type\":\"text\",\"marks\":[{\"type\":\"bold\"}],\"text\":\"Overview\"}]},{\"type\":\"paragraph\",\"a

In [4]:
# Key presence and null analysis
all_keys = set()
for item in content_items:
    all_keys.update(item.keys())

n = len(content_items)
print(f"Total distinct keys in content records: {len(all_keys)}")
print()
print(f"{'Key':<40} {'Present':>10} {'Null':>10} {'Non-empty':>10}")
print("-" * 76)
for k in sorted(all_keys):
    present = sum(1 for s in content_items if k in s)
    nulls = sum(1 for s in content_items if k in s and s.get(k) is None)
    non_empty = sum(1 for s in content_items if s.get(k) not in (None, '', []))
    print(f"{k:<40} {present:>10} {nulls:>10} {non_empty:>10}")

Total distinct keys in content records: 67

Key                                         Present       Null  Non-empty
----------------------------------------------------------------------------
_aggregated_locations                          3107          0       2209
autocomplete                                   3142          0       3142
content_all_day_event                           298          0        298
content_audiences                              3142          0         17
content_authored_by_name                       3142          0       3142
content_body                                   3142          0       3142
content_dismiss_validation_warning             3142       3019        123
content_event_ends_at                           298          0        298
content_event_starts_at                         298          0        298
content_event_timezone_id                      3107       2815        292
content_event_timezone_name                    3113       2815   

### 1.1 `content_type` / `content_sub_type` distribution and correlation

In [5]:
print("=== content_type ===")
for k, v in Counter(s.get("content_type") for s in content_items).most_common():
    print(f"  {repr(k)}: {v}")

print("\n=== content_sub_type ===")
for k, v in Counter(s.get("content_sub_type") for s in content_items).most_common():
    print(f"  {repr(k)}: {v}")

pair_counter = Counter((s.get("content_type"), s.get("content_sub_type")) for s in content_items)
print("\n=== Top type/sub_type pairs ===")
for (ct, st), v in pair_counter.most_common(20):
    print(f"  {ct!r} | {st!r}: {v}")

subtype_to_types = defaultdict(set)
for s in content_items:
    subtype_to_types[s.get("content_sub_type")].add(s.get("content_type"))
conflicts = {sub: types for sub, types in subtype_to_types.items() if sub is not None and len(types) > 1}

print(f"\nSub-types mapping to multiple content_type values: {len(conflicts)}")
if conflicts:
    for sub, types in sorted(conflicts.items(), key=lambda x: len(x[1]), reverse=True):
        print(f"  {sub!r}: {sorted(types)}")

=== content_type ===
  'page': 2813
  'event': 298
  'album': 31

=== content_sub_type ===
  'knowledge': 1724
  'news': 1089
  None: 329

=== Top type/sub_type pairs ===
  'page' | 'knowledge': 1724
  'page' | 'news': 1089
  'event' | None: 298
  'album' | None: 31

Sub-types mapping to multiple content_type values: 0


### 1.2 Core field distributions (candidate `:Content` properties)

In [6]:
print("content_status:", Counter(s.get("content_status") for s in content_items))
print("content_is_featured:", Counter(s.get("content_is_featured") for s in content_items))
print("content_is_must_read:", Counter(s.get("content_is_must_read") for s in content_items))
print("is_deleted:", Counter(s.get("is_deleted") for s in content_items))

validated_non_null = sum(1 for s in content_items if s.get("content_validated_on") not in (None, ""))
print(f"content_validated_on non-null: {validated_non_null}/{len(content_items)}")

agg_non_null = sum(1 for s in content_items if s.get("_aggregated_locations") not in (None, [], ""))
print(f"_aggregated_locations non-empty: {agg_non_null}/{len(content_items)}")

print("\nNOTE: `content_topics` is explicitly deferred from extraction/ingestion for now.")
content_topics_non_empty = sum(1 for s in content_items if s.get("content_topics") not in (None, [], ""))
print(f"content_topics non-empty: {content_topics_non_empty}/{len(content_items)}")

content_status: Counter({'published': 2034, 'unpublished': 851, 'draft': 247, 'rejected': 6, 'pending': 4})
content_is_featured: Counter({False: 3141, True: 1})
content_is_must_read: Counter({False: 3098, None: 35, True: 9})
is_deleted: Counter({False: 2944, True: 198})
content_validated_on non-null: 3107/3142
_aggregated_locations non-empty: 2209/3142

NOTE: `content_topics` is explicitly deferred from extraction/ingestion for now.
content_topics non-empty: 1486/3142


### 1.3 Cross-entity reference checks (`:Site`, `:People`)

In [7]:
site_ref_ids = set(s["content_site_id"] for s in content_items if s.get("content_site_id"))
missing_site_refs = sorted(site_ref_ids - site_ids)
print(f"Unique content_site_id: {len(site_ref_ids)}")
print(f"Missing content_site_id in Site objects: {len(missing_site_refs)}")
if missing_site_refs:
    print("  sample missing:", missing_site_refs[:20])

for fk in ["createdbyid", "lastmodifiedbyid"]:
    refs = set(s[fk] for s in content_items if s.get(fk))
    missing = sorted(refs - people_ids)
    print(f"\n{fk} unique refs: {len(refs)}")
    print(f"{fk} missing in People objects: {len(missing)}")
    if missing:
        print("  sample missing:", missing[:20])

Unique content_site_id: 39
Missing content_site_id in Site objects: 0

createdbyid unique refs: 272
createdbyid missing in People objects: 0

lastmodifiedbyid unique refs: 146
lastmodifiedbyid missing in People objects: 0


### 1.4 Inferred entity analysis: `SitePageCategory` (site-scoped)

In [8]:
# Category pair checks
cat_records = [
    s for s in content_items
    if s.get("content_page_category_id") or s.get("content_page_category_name")
]

id_to_names = defaultdict(set)
name_to_ids = defaultdict(set)
site_scoped_keys = set()

for s in cat_records:
    cid = s.get("content_page_category_id")
    cname = s.get("content_page_category_name")
    sid = s.get("content_site_id")

    if cid is not None:
        id_to_names[cid].add(cname)
    if cname is not None:
        name_to_ids[cname].add(cid)

    # site-scoped identity candidate
    if sid and cid and cname:
        site_scoped_keys.add((sid, cid, cname))

id_conflicts = {cid: names for cid, names in id_to_names.items() if len(names) > 1}
name_conflicts = {name: ids for name, ids in name_to_ids.items() if len(ids) > 1}

missing_both = sum(1 for s in content_items if not s.get("content_page_category_id") and not s.get("content_page_category_name"))
only_id = sum(1 for s in content_items if s.get("content_page_category_id") and not s.get("content_page_category_name"))
only_name = sum(1 for s in content_items if s.get("content_page_category_name") and not s.get("content_page_category_id"))

print(f"Rows with category signal (id or name): {len(cat_records)}")
print(f"Distinct site-scoped (site,id,name) keys: {len(site_scoped_keys)}")
print(f"Category id→name conflicts: {len(id_conflicts)}")
print(f"Category name→id conflicts (global): {len(name_conflicts)}")
print(f"Missing both category id+name: {missing_both}")
print(f"Only id present: {only_id}")
print(f"Only name present: {only_name}")

if name_conflicts:
    print("\nSample name→id conflicts (suggests site scoping is important):")
    for i, (name, ids) in enumerate(list(name_conflicts.items())[:20], 1):
        print(f"  {i:>2}. {name!r} -> {sorted(ids)}")

Rows with category signal (id or name): 2813
Distinct site-scoped (site,id,name) keys: 279
Category id→name conflicts: 0
Category name→id conflicts (global): 20
Missing both category id+name: 329
Only id present: 0
Only name present: 0

Sample name→id conflicts (suggests site scoping is important):
   1. 'Product Best Practices' -> ['75aed90e-3387-4681-8539-51abd306e093', 'bb4152c4-f28d-44e7-a60b-7ee378883bba']
   2. 'Integrations' -> ['a529c6d0-5164-4e91-a4ff-226082c6eb12', 'ee8cd052-7555-40d2-8ec4-e98df91e1d9e']
   3. 'Uncategorized' -> ['02930a2b-c046-4fb7-a98f-fdf9b61cde08', '09134c6e-4347-438b-aeba-f3a9d437462a', '2ef865e8-73ec-40af-bd10-f2045a7030dd', '3868f891-ecac-470f-8eeb-bc93a2a13c4a', '3947fda1-c39e-4552-a524-3bdf6751e782', '3d514093-a349-4f1f-a0c1-02a6c6e6d4b0', '43de2b38-e9d5-412e-9ea8-81ccbb16f1f5', '4754215e-399f-4bc1-b2d0-b6ecb80612e1', '48b165f0-17e8-4675-855f-de48d10ed670', '4e0b9f67-8adc-42c1-a3bb-bd2d08befade', '53d31727-6a45-492b-aabb-5b73f2993b4e', '62cd6243-5835

### 1.5 `content_files` mapping and reconciliation with non-image files

In [9]:
# Load normalized file IDs + file_content_mappings from updated file pipeline output
normalized_file_ids = set()
normalized_file_to_content_ids = defaultdict(set)   # file_id -> {content_id}
normalized_content_to_file_ids = defaultdict(set)   # content_id -> {file_id}

norm_file_path = NORMALIZED_DIR / "files.jsonl"
if norm_file_path.exists():
    with open(norm_file_path) as f:
        for line in f:
            obj = json.loads(line)
            file_id = obj.get("id")
            if file_id:
                normalized_file_ids.add(file_id)

            # IMPORTANT: file_content_mappings contains CONTENT ids (not file ids)
            for content_id in (obj.get("file_content_mappings") or []):
                if content_id and file_id:
                    normalized_file_to_content_ids[file_id].add(content_id)
                    normalized_content_to_file_ids[content_id].add(file_id)

print(f"Normalized non-image file IDs available: {len(normalized_file_ids)}")
print(f"Files with non-empty file_content_mappings: {len(normalized_file_to_content_ids)}")
print(f"Distinct content IDs referenced by file_content_mappings: {len(normalized_content_to_file_ids)}")

# Validate mapped content IDs against actual raw content IDs
content_ids = {s.get("id") for s in content_items if s.get("id")}
mapped_content_ids = set(normalized_content_to_file_ids.keys())

present_content_refs = mapped_content_ids & content_ids
missing_content_refs = mapped_content_ids - content_ids

print(f"Mapped content IDs present in raw content objects: {len(present_content_refs)}")
print(f"Mapped content IDs missing from raw content objects: {len(missing_content_refs)}")

Normalized non-image file IDs available: 555
Files with non-empty file_content_mappings: 296
Distinct content IDs referenced by file_content_mappings: 233
Mapped content IDs present in raw content objects: 22
Mapped content IDs missing from raw content objects: 211


In [10]:
# content_files structure inspection
shape_counter = Counter()
shape_examples = {}

for s in content_items:
    cf = s.get("content_files")
    if isinstance(cf, list):
        if not cf:
            key = "list_empty"
        else:
            key = f"list_of_{type(cf[0]).__name__}"
        shape_counter[key] += 1
        shape_examples.setdefault(key, cf[:2])
    elif cf is None:
        shape_counter["none"] += 1
    else:
        key = type(cf).__name__
        shape_counter[key] += 1
        shape_examples.setdefault(key, cf)

print("content_files shapes:")
for k, v in shape_counter.items():
    print(f"  {k}: {v}")

print("\nSample content_files payloads:")
for k, v in shape_examples.items():
    print(f"  {k}: {v}")

content_files shapes:
  list_of_dict: 1144
  list_empty: 1980
  none: 18

Sample content_files payloads:
  list_of_dict: [{'file_id': '26b4b697-62e1-4e20-b424-3fde3c5ea6ab', 'file_name': 'mceclip1.png', 'file_url': '/content/r/26b4b697-62e1-4e20-b424-3fde3c5ea6ab', 'file_thumbnail_url': None, 'file_mime_type': 'image/png', 'file_type': 'PNG'}]
  list_empty: []


In [11]:
# Extract file refs from content_files (these are FILE ids)
content_file_refs = []
content_file_ref_details = {}

for s in content_items:
    cf = s.get("content_files")
    if not isinstance(cf, list):
        continue
    for item in cf:
        if isinstance(item, str):
            if item:
                content_file_refs.append(item)
                content_file_ref_details.setdefault(item, {"source": "str", "sample": item})
        elif isinstance(item, dict):
            fid = item.get("file_id") or item.get("content_file_id")
            if fid:
                content_file_refs.append(fid)
                content_file_ref_details.setdefault(
                    fid,
                    {
                        "source": "dict",
                        "file_name": item.get("file_name"),
                        "file_type": item.get("file_type"),
                        "file_mime_type": item.get("file_mime_type"),
                        "file_url": item.get("file_url"),
                    },
                )

content_file_ref_set = set(content_file_refs)

present_in_normalized = content_file_ref_set & normalized_file_ids
missing_in_normalized = content_file_ref_set - normalized_file_ids

missing_that_are_raw_images = missing_in_normalized & raw_image_file_ids
missing_that_are_raw_non_images = missing_in_normalized & raw_non_image_file_ids
missing_not_in_any_raw_file = missing_in_normalized - raw_all_file_ids

print("=== content_files -> file ID validation ===")
print(f"Total refs extracted                         : {len(content_file_refs)}")
print(f"Unique file IDs from content_files          : {len(content_file_ref_set)}")
print(f"Present in normalized file IDs              : {len(present_in_normalized)}")
print(f"Missing in normalized file IDs              : {len(missing_in_normalized)}")
print(f"  ├─ of which are image file IDs in raw     : {len(missing_that_are_raw_images)}")
print(f"  ├─ of which are non-image file IDs in raw : {len(missing_that_are_raw_non_images)}")
print(f"  └─ not found in any raw file IDs          : {len(missing_not_in_any_raw_file)}")

if missing_that_are_raw_non_images:
    print("\n⚠️ Unexpected: these IDs are non-image raw files but missing in normalized files")
    print(f"Sample: {sorted(missing_that_are_raw_non_images)[:25]}")

if missing_not_in_any_raw_file:
    print("\n⚠️ Unexpected: these IDs do not exist in raw file object IDs")
    missing_sample = sorted(missing_not_in_any_raw_file)[:10]
    print(f"ID sample: {missing_sample}")
    print("Payload sample from content_files for those IDs:")
    for fid in missing_sample:
        print(f"  {fid}: {content_file_ref_details.get(fid)}")

=== content_files -> file ID validation ===
Total refs extracted                         : 2910
Unique file IDs from content_files          : 2899
Present in normalized file IDs              : 111
Missing in normalized file IDs              : 2788
  ├─ of which are image file IDs in raw     : 2249
  ├─ of which are non-image file IDs in raw : 0
  └─ not found in any raw file IDs          : 539

⚠️ Unexpected: these IDs do not exist in raw file object IDs
ID sample: ['00285705-17ec-4bd3-9584-b692eaf490e9', '00a5b5e3-b832-4c76-b63d-60852c3a8452', '00e64d95-18c3-40a3-b61e-45abc2834874', '018367c7-8065-4a71-983e-4c04f66d0b8b', '01c65ecd-f8ad-42eb-9006-37e6802c1d7c', '02b4063b-36c8-47d4-8d46-efd179f6b6cb', '0325e4a6-b3c6-468a-b340-34bc9ba7558e', '04f6b836-e87e-4a18-901e-2a0d1b4f39e8', '058d82d5-71e1-4e74-a24e-89c4d2b70274', '05950a01-0189-4bee-975e-eb39230ca2f3']
Payload sample from content_files for those IDs:
  00285705-17ec-4bd3-9584-b692eaf490e9: {'source': 'dict', 'file_name': 'India I

In [12]:
# Compare FILE IDs from content_files against FILE IDs implied by file_content_mappings
content_ids = set(s.get("id") for s in content_items if s.get("id"))

file_ids_from_file_content_mappings = set()
for content_id in content_ids:
    file_ids_from_file_content_mappings.update(normalized_content_to_file_ids.get(content_id, set()))

overlap = content_file_ref_set & file_ids_from_file_content_mappings
only_content_files_signal = content_file_ref_set - file_ids_from_file_content_mappings
only_file_content_mappings_signal = file_ids_from_file_content_mappings - content_file_ref_set

print("=== signal comparison: content_files FILE IDs vs inverted file_content_mappings FILE IDs ===")
print(f"File IDs from content_files                               : {len(content_file_ref_set)}")
print(f"File IDs from file_content_mappings inversion (normalized): {len(file_ids_from_file_content_mappings)}")
print(f"Overlap                                                   : {len(overlap)}")
print(f"Only in content_files signal                              : {len(only_content_files_signal)}")
print(f"Only in file_content_mappings signal                      : {len(only_file_content_mappings_signal)}")

=== signal comparison: content_files FILE IDs vs inverted file_content_mappings FILE IDs ===
File IDs from content_files                               : 2899
File IDs from file_content_mappings inversion (normalized): 51
Overlap                                                   : 17
Only in content_files signal                              : 2882
Only in file_content_mappings signal                      : 34


### 1.6 Reverse link check from file side (`file_content_mappings`)

In [13]:
# Reverse linkage using normalized file field: file_content_mappings
# IMPORTANT: file_content_mappings holds CONTENT IDs, so this section validates content-id coverage.
content_ids = set(s.get("id") for s in content_items if s.get("id"))

mapped_content_ids_from_files = set(normalized_content_to_file_ids.keys())
matched = mapped_content_ids_from_files & content_ids
missing_in_content = sorted(mapped_content_ids_from_files - content_ids)
content_without_file_mapping = sorted(content_ids - mapped_content_ids_from_files)

print(f"Distinct content IDs referenced by files.file_content_mappings: {len(mapped_content_ids_from_files)}")
print(f"Referenced IDs matching content.id: {len(matched)}")
print(f"Referenced IDs missing in content set: {len(missing_in_content)}")
if missing_in_content:
    print(f"Missing sample: {missing_in_content[:25]}")

print(f"\nContent IDs with no file_content_mappings back-reference: {len(content_without_file_mapping)}")
print(f"Sample content IDs without mapping: {content_without_file_mapping[:25]}")

print("\nInterpretation:")
print("- file_content_mappings is the canonical normalized reverse signal from files -> content.")
print("- It should be interpreted as content IDs (not file IDs).")
print("- For content ingestion, safest approach is to union file edges from:")
print("  (a) content.content_files[*].file_id and (b) inverted files.file_content_mappings.")
print("- Then keep only file IDs that exist in non-image normalized files and deduplicate edges.")

Distinct content IDs referenced by files.file_content_mappings: 233
Referenced IDs matching content.id: 22
Referenced IDs missing in content set: 211
Missing sample: ['100421', '102895', '103056', '103070', '103087', '103283', '104497', '104518', '105566', '107959', '108949', '108953', '108974', '108987', '109042', '109050', '119903', '119965', '124680', '13802', '13963', '14127', '14469', '14492', '14495']

Content IDs with no file_content_mappings back-reference: 3120
Sample content IDs without mapping: ['00136183-9964-41cc-84f3-4c54917a06ae', '001da714-3447-45e8-b124-274fed463599', '0065caa3-a922-4228-9994-648e8b2c987a', '0078d557-e91c-4210-8d35-61353641470c', '0081f42c-37b7-4e8b-ac55-fb3fdbb798c3', '008b37b3-7e13-49ed-94e0-c7495a11fd5a', '008ec1ee-6cdc-44f0-80d3-4ce3699eae4f', '0099f02f-88e8-4653-bbd8-bea9986885d5', '00a45f2e-372b-487b-9593-b9004e0fb03a', '00bbff09-6620-4918-9465-e01700ded640', '00c5a078-5972-4500-b9ae-e9e721cc6552', '00cd1d58-c5dc-413d-b241-c77a6de719a1', '00cdc88

### 1.7 DataFrame overview (core fields)

In [14]:
core_cols = [
    "id", "content_title", "content_type", "content_sub_type",
    "content_status", "content_site_id", "content_page_category_id",
    "createdbyid", "lastmodifiedbyid", "createddate", "lastmodifieddate"
]
df = pd.DataFrame([{k: s.get(k) for k in core_cols} for s in content_items])
print(df.shape)
df.head(5)

(3142, 11)


,id,content_title,content_type,content_sub_type,content_status,content_site_id,content_page_category_id,createdbyid,lastmodifiedbyid,createddate,lastmodifieddate
0,03e5a566-4785-4e48-8b07-23698ff5cf11,Add-On Products: Recognition Best Practice,page,knowledge,unpublished,906512a7-0e25-40f5-9ace-375b5a7b59e5,bb4152c4-f28d-44e7-a60b-7ee378883bba,9a8faf20-5bcf-4fef-929f-fcd4e1283b4d,e8bcab12-2978-4c65-a511-8d6d62c27102,2024-12-19T18:12:00.285Z,2025-03-26T22:33:58.825Z
1,03e890aa-0654-46f4-b0ec-464d0e715bf2,Best Practices: Segments (SFDC),page,knowledge,unpublished,906512a7-0e25-40f5-9ace-375b5a7b59e5,bb4152c4-f28d-44e7-a60b-7ee378883bba,acc88c24-7434-47d7-bd00-69b0890b9af8,e9114dcb-819d-406a-b177-b0d7dfe805fd,2024-04-18T18:54:27.220Z,2025-03-26T22:50:02.307Z
2,03eb64e8-1e46-41ab-9c18-995faf0ac087,Pro Tip: Win Reports in Backyard,page,knowledge,published,7db47563-7c6a-4206-bf10-3e736f7d2804,378c108e-40dc-4470-a9c1-5013ee7576e0,df2bea29-6956-42bf-9c2f-67cf081f39da,df2bea29-6956-42bf-9c2f-67cf081f39da,2024-08-06T20:01:10.326Z,2025-12-05T02:11:11.508Z
3,03f48976-1a5f-4624-b8de-dd2d2c3795bf,"LumApps Webinar ""Measuring the Impact of Your ...",page,knowledge,published,39b3d853-fcc5-43cd-a508-d77f56bb0b02,ddbc8032-132f-44fc-9ed6-4b4803fa50b1,3a331b59-add7-4198-a9a8-78c11af00261,3a331b59-add7-4198-a9a8-78c11af00261,2024-08-26T20:01:45.645Z,2025-07-30T19:19:45.369Z
4,04060b92-c2be-4f45-a802-6dd3c198f89a,Chatbot Integration- Aisera,page,knowledge,published,73746744-c930-4597-9c7d-9723c18c7d07,ee8cd052-7555-40d2-8ec4-e98df91e1d9e,86e941b2-44d3-4424-a637-809d0122d5a6,3f1dcf21-0489-46de-a743-1a47b506e795,2024-03-26T19:15:31.948Z,2025-11-19T12:45:50.362Z


---
## 🔍 Exploration Summary

### What we learned
- **3,142** content records; structurally richer than people/site/file.
- `content_type` and `content_sub_type` are strongly correlated and conflict-free.
- `content_site_id` / `createdbyid` / `lastmodifiedbyid` references are clean — zero missing.
- `content_files` is a nested list-of-dicts (`file_id`, etc.); most point to image files excluded in the file pipeline.
- File pipeline emits unified `file_content_mappings` in `files.jsonl`; content↔file reconciliation uses union of both signals, deduped via MERGE.
- Page category name collisions exist globally; modeling as **site-scoped `SitePageCategory`** is justified.
- `_aggregated_locations` retained as-is.
- `content_topics` deferred — ignored in initial extraction/ingestion.
- `parent_content_id` ignored — only 2/3,142 records have it; `content_is_parent` is True for 99.94% (meaningless flag).

### Design decisions (confirmed)
1. **`:Content` node properties:** `id`, `content_title`, `content_type`, `content_sub_type`, `content_status`, `content_is_featured`, `content_is_must_read`, `is_deleted`, `_aggregated_locations`, `createddate`, `lastmodifieddate`, `content_first_published_on`, `content_publish_on_date`, `content_validated_on`
2. **`LAST_MODIFIED`:** Yes — `(People)-[:LAST_MODIFIED]->(Content)` from `lastmodifiedbyid`
3. **`SitePageCategory` identity:** `page_category_id` (UUID) as unique key, with `site_id` and `name` as properties
4. **Null category handling:** Skip — no `HAS_CONTENT` edge when `content_page_category_id` is null
5. **File linkage:** Union of both signals (`content_files[*].file_id` + inverted `file_content_mappings`), deduped via MERGE
6. **Parent hierarchy:** Ignored — only 2/3,142 records; not worth the complexity

---
## 2 · Extract & Normalise

In [15]:
from src.extractors.content_extractor import extract_contents, write_normalized

result = extract_contents(RAW_FILE)

print(f"Contents extracted         : {len(result['contents'])}")
print(f"Page categories (unique)   : {len(result['page_categories'])}")
print(f"Contents without site      : {result['num_no_site']}")
print(f"Contents without category  : {result['num_no_category']}")
print(f"Contents with file refs    : {result['num_with_files']}")

write_normalized(result, NORMALIZED_DIR)

Contents extracted         : 3142
Page categories (unique)   : 279
Contents without site      : 0
Contents without category  : 329
Contents with file refs    : 1144
Written 3142 content records → /home/ubuntu/graph_experiments/data/normalized/contents.jsonl
Written 279 page category records → /home/ubuntu/graph_experiments/data/normalized/content_page_categories.jsonl


## 3 · Analyse Extraction Results

In [16]:
# Load normalized outputs
contents_norm = []
with open(NORMALIZED_DIR / "contents.jsonl") as f:
    for line in f:
        contents_norm.append(json.loads(line))

cats_norm = []
with open(NORMALIZED_DIR / "content_page_categories.jsonl") as f:
    for line in f:
        cats_norm.append(json.loads(line))

print(f"contents.jsonl rows                 : {len(contents_norm)}")
print(f"content_page_categories.jsonl rows  : {len(cats_norm)}")

print("\nContents columns:")
print(sorted(contents_norm[0].keys()))

contents.jsonl rows                 : 3142
content_page_categories.jsonl rows  : 279

Contents columns:
['_aggregated_locations', 'content_file_ids', 'content_first_published_on', 'content_is_featured', 'content_is_must_read', 'content_page_category_id', 'content_page_category_name', 'content_publish_on_date', 'content_site_id', 'content_status', 'content_sub_type', 'content_title', 'content_type', 'content_validated_on', 'createdbyid', 'createddate', 'id', 'is_deleted', 'lastmodifiedbyid', 'lastmodifieddate']


In [17]:
# FK validation against raw entity sets
creator_ids_norm = set(r["createdbyid"] for r in contents_norm if r.get("createdbyid"))
modifier_ids_norm = set(r["lastmodifiedbyid"] for r in contents_norm if r.get("lastmodifiedbyid"))
site_ids_norm = set(r["content_site_id"] for r in contents_norm if r.get("content_site_id"))

missing_creators = creator_ids_norm - people_ids
missing_modifiers = modifier_ids_norm - people_ids
missing_sites = site_ids_norm - site_ids

print(f"Unique createdbyid refs   : {len(creator_ids_norm)}")
print(f"Missing createdbyid refs  : {len(missing_creators)}")
print(f"Unique lastmodifiedbyid   : {len(modifier_ids_norm)}")
print(f"Missing lastmodifiedbyid  : {len(missing_modifiers)}")
print(f"Unique content_site_id    : {len(site_ids_norm)}")
print(f"Missing content_site_id   : {len(missing_sites)}")

num_no_cat = sum(1 for r in contents_norm if not r.get("content_page_category_id"))
print(f"Contents without category : {num_no_cat}")

Unique createdbyid refs   : 272
Missing createdbyid refs  : 0
Unique lastmodifiedbyid   : 146
Missing lastmodifiedbyid  : 0
Unique content_site_id    : 39
Missing content_site_id   : 0
Contents without category : 329


## 4 · Ingest into Neo4j

In [18]:
from src.loaders.content_loader import load_content_graph
from src.utils.neo4j_client import Neo4jClient

client = Neo4jClient(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
load_content_graph(client, NORMALIZED_DIR, batch=True)

✅ Constraints ensured.
✅ SitePageCategory nodes loaded: 279
✅ Content nodes loaded: 3142
✅ BELONGS_TO_SITE relationships: 3142
✅ CREATED relationships: 3142
✅ LAST_MODIFIED relationships: 3142
✅ HAS_CONTENT relationships: 2813
✅ SitePageCategory BELONGS_TO_SITE relationships: 279
✅ HAS_FILE relationships (from content_files): 2910 pairs attempted
✅ HAS_FILE relationships (from file_content_mappings): 296 pairs attempted

✅ Content graph fully loaded.


## 5 · Post-Ingestion Verification

In [19]:
# Node counts
node_counts = client.query("""
    MATCH (n)
    WHERE n:Content OR n:SitePageCategory OR n:People OR n:Site OR n:File
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY label
""")
print("=== Node counts ===")
for row in node_counts:
    print(f"  :{row['label']:<20} {row['count']:>6}")

print()

# Relationship counts
rel_counts = client.query("""
    MATCH ()-[r]->()
    WHERE type(r) IN ['BELONGS_TO_SITE', 'CREATED', 'LAST_MODIFIED', 'HAS_CONTENT', 'HAS_FILE']
    RETURN type(r) AS rel_type, count(r) AS count
    ORDER BY rel_type
""")
print("=== Relationship counts ===")
for row in rel_counts:
    print(f"  {row['rel_type']:<25} {row['count']:>6}")

=== Node counts ===
  :Content                3142
  :File                    555
  :People                  954
  :Site                     45
  :SitePageCategory        278

=== Relationship counts ===
  BELONGS_TO_SITE             3982
  CREATED                     3742
  HAS_CONTENT                 2813
  HAS_FILE                     150
  LAST_MODIFIED               3187


In [20]:
print("=== content_type distribution in Neo4j ===")
rows = client.query("""
    MATCH (c:Content)
    RETURN coalesce(c.content_type, '(null)') AS content_type, count(c) AS cnt
    ORDER BY cnt DESC
""")
for row in rows:
    print(f"  {row['content_type']:<20} {row['cnt']:>5}")

print()
print("=== Top 10 Sites by SitePageCategory count ===")
rows = client.query("""
    MATCH (spc:SitePageCategory)-[:BELONGS_TO_SITE]->(s:Site)
    RETURN s.site_name AS site, count(spc) AS cat_count
    ORDER BY cat_count DESC
    LIMIT 10
""")
for row in rows:
    print(f"  {row['site']:<40} {row['cat_count']:>5}")

print()
print("=== Top 10 People by content CREATED ===")
rows = client.query("""
    MATCH (p:People)-[:CREATED]->(c:Content)
    RETURN p.user_name AS person, count(c) AS content_count
    ORDER BY content_count DESC
    LIMIT 10
""")
for row in rows:
    print(f"  {row['person']:<40} {row['content_count']:>5}")

rows = client.query("""
    MATCH (spc:SitePageCategory)-[:HAS_CONTENT]->(c:Content)
    RETURN count(*) AS has_content_count
""")
print(f"\nHAS_CONTENT relationships created: {rows[0]['has_content_count']}")

rows = client.query("""
    MATCH (c:Content)-[:HAS_FILE]->(f:File)
    RETURN count(*) AS has_file_count
""")
print(f"HAS_FILE relationships created: {rows[0]['has_file_count']}")

=== content_type distribution in Neo4j ===
  page                  2813
  event                  298
  album                   31

=== Top 10 Sites by SitePageCategory count ===
  User Research                               25
  Revenue Team                                20
  Product Team                                19
  Engineering Team                            14
  Company                                     14
  Customer Experience Team                    12
  Customer Success Managers                   11
  Life at Simpplr                             10
  Professional Services                       10
  Solutions Consulting                        10

=== Top 10 People by content CREATED ===
  Madhavi Polepeddi                          465
  Christine Robertson                        205
  Kaitlin Stouter                            203
  Christy Schoon                             180
  Rachel Smith                               115
  Talissa Davis                              

In [21]:
client.close()
print("Neo4j connection closed.")

Neo4j connection closed.
